# Fase 4 v2 — 03: video con minimap (camino C, SAM3 masks)

Genera el video con **minimap cenital + trayectorias** usando `render_minimap_video`
(camino C): SAM3 segmenta `green_floor` (alfombra → líneas blancas) y `yellow_zone`/
`blue_zone` (porterías → orientación), detecta robots/balón, calcula la homografía
por frame y dibuja el panel cenital con estelas.

**Requiere GPU + SAM3** (`assets/sam3`) — correr en el pod. Usa `frame_step`/`max_frames`
para acotar el costo. Devuelve conteo de homografía `estimated/propagated/rejected`.

In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.core.minimap_pipeline import render_minimap_video

OUT = REPO / 'notebooks/fase_4_homografia/outputs/videos'
OUT.mkdir(parents=True, exist_ok=True)
print('REPO:', REPO)

In [ ]:
VIDEO = '/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV'
raw = OUT / 'minimap_cenital_9933.mp4'

res = render_minimap_video(
    VIDEO,
    output_path=str(raw),
    start_frame=0, frame_step=3, max_frames=120,
    detector='sam3_text',   # green_floor + yellow_zone/blue_zone via SAM3
    draw_overlay=True,      # dibuja el template proyectado sobre el frame
    progress=True,
)
print('homografia:', res['homography'])

# transcode h264 yuv420p (compatibilidad de reproductores/navegador)
h264 = OUT / 'minimap_cenital_9933_h264.mp4'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(raw),
                '-vcodec','libx264','-pix_fmt','yuv420p',str(h264)], check=True)
print('video h264:', h264, round(os.path.getsize(h264)/1e6,2),'MB')

## Notas

- `homography.rejected` alto = la cámara cenital panea y el gate de consistencia
  descarta saltos → la H se propaga (puede quedar corrida, p. ej. círculo central
  desfasado). Mejoras en curso: esquinas tolerantes a oclusión
  (`homography_multifeature.inner_corners_extrapolated`) + fit multi-feature.
- Para lateral, `detector='sam3_text'` no segmenta bien `blue_zone`; usar `yolo_sam3`
  cuando aplique. La robustez lateral es el siguiente foco.